# Single Point

Here is an example of loading up the baseline simualtion and returning information about the visits that overlap a single point in the sky.

Installation instructions for rubin_sim can be found in the documentation at https://rubin-sim.lsst.io

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-04
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/tutorial

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
import healpy as hp

import rubin_sim.maf as maf
import rubin_scheduler.utils as rsUtils
from rubin_sim.data import get_baseline

In [ ]:
# Grab the current baseline file. Should have been downloaded with rubin_sim.
# Can grab lots of different sims from: http://astro-lsst-01.astro.washington.edu:8081/
baseline_file = get_baseline()
name = os.path.basename(baseline_file).replace(".db", "")

data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="05_single_point_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

out_dir = data_dir
results_db = maf.db.ResultsDb(out_dir=out_dir)

In [ ]:
bundle_list = []
# The point on the sky we would like to get visits for
ra = [0.0]
dec = [-20]


# Say we just want to pass data through, not compute anything. Documentation on
# columns at:  https://rubin-sim.lsst.io/rs_scheduler/output_schema.html
metric = maf.metrics.PassMetric(cols=["band", "observationStartMJD", "fiveSigmaDepth", "visitExposureTime"])
# Select all the visits. Could be something like "filter='r'", "night < 365", etc
sql = ""
slicer = maf.slicers.UserPointsSlicer(ra=ra, dec=dec)
bundle_list.append(maf.MetricBundle(metric, slicer, sql, run_name=name))

In [ ]:
bd = maf.metricBundles.make_bundles_dict_from_list(bundle_list)
bg = maf.metricBundles.MetricBundleGroup(bd, baseline_file, out_dir=out_dir, results_db=results_db)
bg.run_all()

In [ ]:
# Our bundleList now has values. The trailing [0] is to get the first result. If we specified more
# ra,dec point, those results would be in the later indices.
bundle_list[0].metric_values[0][0:10]

In [ ]:
# As a bit of foreshadowing for how the rest of MAF works, we'll call the visits overlapping a
# single point in the sky "dataSlice".
data_slice = bundle_list[0].metric_values[0]

# Let's plot up what the 5-sigma depth looks like
plt.figure()
# Give each filter it's own color
f2c = {"u": "purple", "g": "blue", "r": "green", "i": "cyan", "z": "orange", "y": "red"}
for fn in f2c:
    in_filt = np.where(data_slice["band"] == fn)[0]
    plt.plot(
        data_slice["observationStartMJD"][in_filt],
        data_slice["fiveSigmaDepth"][in_filt],
        "o",
        color=f2c[fn],
        label=fn + r", $\sigma=$ %.2f" % (np.std(data_slice["fiveSigmaDepth"][in_filt])),
        alpha=0.5,
    )
plt.xlabel("MJD (days)")
plt.ylabel(r"5$\sigma$ depth (mags)")
plt.legend(loc=(1.04, 0))
plt.title("%s\nObservations at ra=%.3f, dec=%.3f" % (name, ra[0], dec[0]))

In [ ]:
n_vis = np.arange(data_slice["observationStartMJD"].size)
mjd = np.sort(data_slice["observationStartMJD"])
plt.plot(mjd, n_vis)
plt.title("%s\nObservations at ra=%.3f, dec=%.3f" % (name, ra[0], dec[0]))
plt.xlabel("MJD (days)")
plt.ylabel("Cummulative Number of Visits")

In [ ]:
# Note there are some odd low-depth outliers. Those are probably the 1x15s
# twilight time exposures. let's try filtering those out

data_slice = bundle_list[0].metric_values[0]

# crop off short exposure times
data_slice = data_slice[np.where(data_slice["visitExposureTime"] > 25.0)]

# Let's plot up what the 5-sigma depth looks like
plt.figure()
# Give each filter it's own color
f2c = {"u": "purple", "g": "blue", "r": "green", "i": "cyan", "z": "orange", "y": "red"}
for fn in f2c:
    in_filt = np.where(data_slice["band"] == fn)[0]
    plt.plot(
        data_slice["observationStartMJD"][in_filt],
        data_slice["fiveSigmaDepth"][in_filt],
        "o",
        color=f2c[fn],
        label=fn
        + r", N=%i, $\sigma=$ %.2f" % (np.size(in_filt), np.std(data_slice["fiveSigmaDepth"][in_filt])),
        alpha=0.5,
    )
plt.xlabel("MJD (days)")
plt.ylabel(r"5$\sigma$ depth (mags)")
plt.legend(loc=(1.04, 0))
plt.title("%s\nObservations at ra=%.3f, dec=%.3f" % (name, ra[0], dec[0]))

In [ ]:
# and now looking only at short exposure times

data_slice = bundle_list[0].metric_values[0]

# Include only short exposures
data_slice = data_slice[np.where(data_slice["visitExposureTime"] < 25.0)]

# Let's plot up what the 5-sigma depth looks like
plt.figure()
# Give each filter it's own color
f2c = {"u": "purple", "g": "blue", "r": "green", "i": "cyan", "z": "orange", "y": "red"}
for fn in f2c:
    in_filt = np.where(data_slice["band"] == fn)[0]
    plt.plot(
        data_slice["observationStartMJD"][in_filt],
        data_slice["fiveSigmaDepth"][in_filt],
        "o",
        color=f2c[fn],
        label=fn + r", $\sigma=$ %.2f" % (np.std(data_slice["fiveSigmaDepth"][in_filt])),
        alpha=0.5,
    )
plt.xlabel("MJD (days)")
plt.ylabel(r"5$\sigma$ depth (mags)")
plt.legend(loc=(1.04, 0))
plt.title("%s\nObservations at ra=%.3f, dec=%.3f" % (name, ra[0], dec[0]))

Let's try out doing it on a DDF location

In [ ]:
bundle_list = []
# The point on the sky we would like to get visits for
ra = [9.450]
dec = [-44.0]


# Say we just want to pass data through, not compute anything. Documentation on
# columns at:  https://rubin-sim.lsst.io/rs_scheduler/output_schema.html
metric = maf.metrics.PassMetric(cols=["band", "observationStartMJD", "fiveSigmaDepth", "visitExposureTime"])
# Select all the visits. Could be something like "filter='r'", "night < 365", etc
sql = ""
slicer = maf.slicers.UserPointsSlicer(ra=ra, dec=dec)
bundle_list.append(maf.MetricBundle(metric, slicer, sql, run_name=name))

bd = maf.metricBundles.make_bundles_dict_from_list(bundle_list)
bg = maf.metricBundles.MetricBundleGroup(bd, baseline_file, out_dir=out_dir, results_db=results_db)
bg.run_all()

In [ ]:
data_slice = bundle_list[0].metric_values[0]

# crop off short exposure times
data_slice = data_slice[np.where(data_slice["visitExposureTime"] > 25.0)]

# Let's plot up what the 5-sigma depth looks like
plt.figure()
# Give each filter it's own color
f2c = {"u": "purple", "g": "blue", "r": "green", "i": "cyan", "z": "orange", "y": "red"}
for fn in f2c:
    in_filt = np.where(data_slice["band"] == fn)[0]
    plt.plot(
        data_slice["observationStartMJD"][in_filt],
        data_slice["fiveSigmaDepth"][in_filt],
        "o",
        color=f2c[fn],
        label=fn
        + r", N=%i, $\sigma=$ %.2f" % (np.size(in_filt), np.std(data_slice["fiveSigmaDepth"][in_filt])),
        alpha=0.1,
    )
plt.xlabel("MJD (days)")
plt.ylabel(r"5$\sigma$ depth (mags)")
plt.legend(loc=(1.04, 0))
plt.title("%s\nObservations at ra=%.3f, dec=%.3f" % (name, ra[0], dec[0]))